# INF201 – Deliverable 2
## Campus Coffee Shop Sales Pipeline

This notebook contains the implementation, testing, and analysis
of the five tasks in Deliverable 2.

## Task 1: File Ingestion and Regular Expressions

### Task 1.1: Robust File Reader

The goal of this task is to read log files that may use different
character encodings. The function first attempts UTF-8 decoding
and falls back to Latin-1 if necessary.

In [64]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import json

In [65]:
def read_log_file(file_path: str | Path) -> tuple[str, str]:
    """Read a log file and return its content and encoding."""
    file_path = Path(file_path)

    # Try reading the file using UTF-8 encoding
    try:
        content = file_path.read_text(encoding="utf-8")
        return content, "utf-8"

    # Fall back to Latin-1 if UTF-8 decoding fails
    except UnicodeDecodeError:
        content = file_path.read_text(encoding="iso-8859-1")
        return content, "iso-8859-1"

In [66]:
# Test the function using a generated log file
test_file = Path("data/2026/library_cafe/POS_01/register_sales.log")

content, encoding = read_log_file(test_file)

print(f"Encoding used: {encoding}")

# Display the first five lines
for line in content.splitlines()[:5]:
    print(line)

Encoding used: utf-8
[2026-10-15 07:03:00] [POS_01] item=Croissant price=3.79 USD (CANCELLED)
--- PRINTER PAPER LOW ---
[2026-10-15 07:05:40] [POS_01] item=Croissant price=4.28 EUR (SUCCESS)
[2026-10-15 07:06:28] [POS_01] item=Croissant price=3.35 USD (SUCCESS)
[DEBUG] BARCODE SCANNER REBOOTED


## Discussion:

The function successfully reads the log file using UTF-8 encoding.
If UTF-8 decoding fails, it attempts Latin-1.

Since ASCII characters are compatible with both encodings,
the original encoding cannot always be identified from the
file content alone.

### Task 1.2: Log Parser Using Regular Expressions

The goal of this task is to extract valid transaction records from the log files using regular expressions (`re`).

Each valid transaction contains six fields: timestamp, register ID, item, price, currency, and status.

System messages and other invalid lines are ignored. The function returns a list of dictionaries containing the extracted transaction data.

In [67]:
def parse_log(content: str) -> list[dict]:
    """Extract valid transactions from log text using regex."""

    # Define a regex pattern with six named capturing groups
    pattern = (
        r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
        r"\[(?P<register_id>\w+)\] "
        r"item=(?P<item>\w+) "
        r"price=(?P<price>\d+\.\d{2}) "
        r"(?P<currency>EUR|NOK|USD) "
        r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
    )

    transactions = []

    # Process the log file one line at a time
    for line in content.splitlines():

        # Check whether the line matches the transaction format
        match = re.match(pattern, line)

        if match:
            # Convert captured groups into a dictionary
            transaction = match.groupdict()

            # Convert the price from string to float
            transaction["price"] = float(transaction["price"])

            # Add the valid transaction to our list
            transactions.append(transaction)

    return transactions

In [68]:
# Parse the content read in Task 1.1
transactions = parse_log(content)

# Display the number of valid transactions
print(f"Number of valid transactions: {len(transactions)}")

# Display the first three parsed transactions
for transaction in transactions[:3]:
    print(transaction)

Number of valid transactions: 80
{'timestamp': '2026-10-15 07:03:00', 'register_id': 'POS_01', 'item': 'Croissant', 'price': 3.79, 'currency': 'USD', 'status': 'CANCELLED'}
{'timestamp': '2026-10-15 07:05:40', 'register_id': 'POS_01', 'item': 'Croissant', 'price': 4.28, 'currency': 'EUR', 'status': 'SUCCESS'}
{'timestamp': '2026-10-15 07:06:28', 'register_id': 'POS_01', 'item': 'Croissant', 'price': 3.35, 'currency': 'USD', 'status': 'SUCCESS'}


## Discussion:

The parser successfully extracted the six required fields from the transaction logs and ignored unrelated system messages. The price values were converted to floats for later numerical analysis.

The parser identified 80 valid transactions in the tested log file.

## Task 2: Recursive Utility Functions

### Task 2.1: Recursive Directory Search

This function recursively searches through nested directories to find
all `.log` files. It uses `pathlib.Path` and explicit recursion without
using `glob()`, `rglob()`, or `os.walk()`.

In [69]:
def find_log_files(directory: str | Path) -> list[Path]:
    """Recursively find all .log files in a directory."""

    directory = Path(directory)
    log_files = []

    # Iterate through all items in the current directory
    for item in directory.iterdir():

        if item.is_dir():
            # Recursively search inside subdirectories
            log_files.extend(find_log_files(item))

        elif item.is_file() and item.suffix == ".log":
            # Add matching log files to the list
            log_files.append(item)

    return log_files

In [70]:
# Search for log files inside the generated data folder
log_files = find_log_files("data")

# Display the total number of log files
print(f"Number of log files found: {len(log_files)}")

# Display all discovered file paths
for file in sorted(log_files):
    print(file)

Number of log files found: 9
data\2026\library_cafe\POS_01\register_sales.log
data\2026\library_cafe\POS_02\register_sales.log
data\2026\library_cafe\POS_03\register_sales.log
data\2026\science_building\POS_01\register_sales.log
data\2026\science_building\POS_02\register_sales.log
data\2026\science_building\POS_03\register_sales.log
data\2026\student_union\POS_01\register_sales.log
data\2026\student_union\POS_02\register_sales.log
data\2026\student_union\POS_03\register_sales.log


## Discussion:

The recursive function successfully searches through the nested
directory structure and finds all nine log files. It uses explicit
recursion as required, without relying on built-in recursive
file-search methods.

### Task 2.2: Extract Active Register IDs

This function recursively traverses nested dictionaries and lists
in the store configuration file to identify active registers.

It returns a set of register IDs where the `active` field is `True`.

In [71]:
def find_active_registers(data: dict | list) -> set[str]:
    """Recursively find active register IDs in nested data."""

    active_registers = set()

    # Check if the current data is a dictionary
    if isinstance(data, dict):

        # Check whether the dictionary represents an active register
        if data.get("active") is True and "id" in data:
            active_registers.add(data["id"])

        # Recursively examine all dictionary values
        for value in data.values():
            if isinstance(value, (dict, list)):
                active_registers.update(find_active_registers(value))

    # Check if the current data is a list
    elif isinstance(data, list):

        # Recursively examine each element in the list
        for item in data:
            if isinstance(item, (dict, list)):
                active_registers.update(find_active_registers(item))

    return active_registers

In [72]:
# Read the store configuration JSON file
config_path = Path("data/store_config.json")

with config_path.open("r", encoding="utf-8") as file:
    store_config = json.load(file)

# Extract active register IDs
active_registers = find_active_registers(store_config)

# Display the results
print(f"Number of active registers: {len(active_registers)}")
print(f"Active register IDs: {sorted(active_registers)}")

Number of active registers: 3
Active register IDs: ['POS_01', 'POS_02', 'POS_03']


## Discussion:

The recursive function identified three active registers from the
nested JSON configuration. The inactive register (POS_04) was excluded.

A set was used to ensure that each register ID appears only once.

## Task 3: Data Cleaning and Analysis Using NumPy

### Task 3.1: Removing Price Outliers

This function identifies price outliers using the three-standard-deviation rule. Outliers are replaced with the median of the valid prices to reduce the effect of extreme values on the analysis.

In [73]:
def remove_price_outliers(prices: list[float] | np.ndarray) -> np.ndarray:
    """Replace price outliers with the median of valid prices."""

    # Convert the input prices to a 1D NumPy float array
    prices = np.asarray(prices, dtype=float).flatten()

    # Calculate the mean and standard deviation
    mean_price = np.mean(prices)
    std_price = np.std(prices)

    # Identify outliers using the three-standard-deviation rule
    outliers = np.abs(prices - mean_price) > 3 * std_price

    # Calculate the median of valid prices
    valid_prices = prices[~outliers]
    median_price = np.median(valid_prices)

    # Replace outliers with the median
    cleaned_prices = prices.copy()
    cleaned_prices[outliers] = median_price

    return cleaned_prices

In [74]:
# Create sample prices with one extreme outlier
test_prices = np.array([4.5] * 20 + [999.0])

# Clean the sample prices
cleaned_prices = remove_price_outliers(test_prices)

# Compare the results
print("Original prices:", test_prices)
print("Cleaned prices:", cleaned_prices)
print("Number of outliers replaced:",
      np.sum(test_prices != cleaned_prices))

Original prices: [  4.5   4.5   4.5   4.5   4.5   4.5   4.5   4.5   4.5   4.5   4.5   4.5
   4.5   4.5   4.5   4.5   4.5   4.5   4.5   4.5 999. ]
Cleaned prices: [4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5 4.5
 4.5 4.5 4.5]
Number of outliers replaced: 1


## Discussion:

The function uses NumPy operations to identify prices that deviate more than three standard deviations from the mean.

Detected outliers are replaced with the median of valid prices rather than removed, preserving the number of transactions.

The method is sensitive to extreme values because outliers also affect the mean and standard deviation.

### Task 3.2: Compute Refund Ratio

This function calculates the refund ratio for each item using NumPy array operations.

A small constant (epsilon = 10⁻⁶) is added to the sales count to prevent division by zero.

In [75]:
def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """Calculate element-wise refund ratios using NumPy."""

    # Small constant to prevent division by zero
    epsilon = 1e-6

    # Calculate refund ratios for all items
    refund_ratios = refund_count / (sales_count + epsilon)

    return refund_ratios

In [76]:
# Sample sales and refund counts for three items
sales_count = np.array([20, 10, 0])
refund_count = np.array([4, 2, 0])

# Calculate refund ratios
refund_ratios = compute_refund_ratio(sales_count, refund_count)

# Display the results
print("Sales count:", sales_count)
print("Refund count:", refund_count)
print("Refund ratios:", np.round(refund_ratios, 4))

Sales count: [20 10  0]
Refund count: [4 2 0]
Refund ratios: [0.2 0.2 0. ]


## Discussion:

The function calculates refund ratios using NumPy's element-wise
array operations without requiring a loop.

Adding epsilon to the denominator prevents division by zero when
the sales count is zero.

## Task 4: Data Analysis Using Pandas

### Task 4.1: Creating a Pandas DataFrame

This function converts parsed transaction records into a Pandas
DataFrame. The timestamp is converted to datetime and used as the
DataFrame index. The transactions are then sorted chronologically.

In [77]:
def create_transaction_dataframe(transactions: list[dict]) -> pd.DataFrame:
    """Create a chronologically sorted DataFrame from transaction records."""

    # Convert the list of dictionaries into a DataFrame
    df = pd.DataFrame(transactions)

    # Convert the timestamp column to datetime
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # Set timestamp as the DataFrame index
    df = df.set_index("timestamp")

    # Sort the transactions chronologically
    df = df.sort_index()

    return df

In [78]:
# Create a DataFrame from the parsed transactions
df = create_transaction_dataframe(transactions)

# Display the first five transactions
print("First five transactions:")
display(df.head())

# Check the DataFrame dimensions and index type
print(f"DataFrame shape: {df.shape}")
print(f"Index type: {type(df.index)}")

First five transactions:


,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:03:00,POS_01,Croissant,3.79,USD,CANCELLED
2026-10-15 07:05:40,POS_01,Croissant,4.28,EUR,SUCCESS
2026-10-15 07:06:28,POS_01,Croissant,3.35,USD,SUCCESS
2026-10-15 07:07:11,POS_01,Latte,4.72,EUR,REFUND
2026-10-15 07:07:47,POS_01,Sandwich,7.49,NOK,REFUND


DataFrame shape: (80, 5)
Index type: <class 'pandas.DatetimeIndex'>


## Discussion:

The parsed transactions were successfully converted into a Pandas
DataFrame with a DatetimeIndex. Sorting by timestamp ensures that
the data is arranged chronologically and ready for time-based analysis.

### Task 4.2: Hourly Revenue Analysis

This function groups transactions by item and resamples them into
one-hour intervals to calculate the total transaction price for
each item per hour.

Pandas `groupby()` and `resample()` are used for this analysis.

In [79]:
def calculate_hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate the total transaction price per item per hour."""

    # Group transactions by item and resample into hourly intervals
    hourly_revenue = (
        df.groupby("item")
        .resample("1h")["price"]
        .sum()
    )

    # Convert the result into a DataFrame
    hourly_revenue = hourly_revenue.reset_index()

    return hourly_revenue

## Test:

In [80]:
# Calculate hourly revenue from the test DataFrame
hourly_revenue = calculate_hourly_revenue(df)

# Display the first ten rows
print("Hourly revenue per item:")
display(hourly_revenue.head(10))

# Check the DataFrame dimensions
print(f"Number of rows: {len(hourly_revenue)}")
print(f"Columns: {hourly_revenue.columns.tolist()}")

Hourly revenue per item:


,item,timestamp,price
0,Cappuccino,2026-10-15 07:00:00,36.94
1,Cappuccino,2026-10-15 08:00:00,1019.84
2,Croissant,2026-10-15 07:00:00,18.89
3,Croissant,2026-10-15 08:00:00,26.64
4,Espresso,2026-10-15 07:00:00,3.21
5,Espresso,2026-10-15 08:00:00,1024.10
6,Latte,2026-10-15 07:00:00,44.42
7,Latte,2026-10-15 08:00:00,44.98
8,Muffin,2026-10-15 07:00:00,13.77
9,Muffin,2026-10-15 08:00:00,18.70


Number of rows: 12
Columns: ['item', 'timestamp', 'price']


## Discussion:

The transactions were grouped by item and resampled into one-hour
intervals. The resulting DataFrame shows the total recorded price
for each product during each hour, providing a basis for analysing
hourly sales patterns.

### Task 4.3: Pivot Table by Transaction Status

This function creates a pivot table to summarize the total
transaction prices for each item across three status categories:
SUCCESS, REFUND, and CANCELLED.

Pandas `pivot_table()` is used to organize and aggregate the data.

In [81]:
def create_status_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """Create a pivot table of total prices by item and status."""

    # Group transaction prices by item and status
    pivot_table = pd.pivot_table(
        df,
        values="price",
        index="item",
        columns="status",
        aggfunc="sum",
        fill_value=0
    )

    return pivot_table

## Test:

In [82]:
# Create the pivot table from our test DataFrame
status_pivot = create_status_pivot_table(df)

# Display the result
print("Transaction prices by item and status:")
display(status_pivot)

# Check the shape
print(f"Pivot table shape: {status_pivot.shape}")

Transaction prices by item and status:


status,CANCELLED,REFUND,SUCCESS
item,,,
Cappuccino,3.80,15.50,1037.48
Croissant,7.42,7.08,31.03
Espresso,3.21,5.96,1018.14
Latte,9.01,13.86,66.53
Muffin,3.89,10.42,18.16
Sandwich,31.03,30.69,45.44


Pivot table shape: (6, 3)


## Discussion:

The pivot table summarizes total transaction prices for each
item across SUCCESS, REFUND, and CANCELLED categories.

Missing item-status combinations are filled with zero.
This provides a clear comparison of transaction values
across different statuses.

### Task 4.4: Refund Ratio per Item

This function calculates the refund ratio for each item based on
the number of SUCCESS and REFUND transactions.

It uses Pandas to count transactions by item and status, and calls
the NumPy function `compute_refund_ratio()` from Task 3.2.

In [83]:
def calculate_item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """Calculate refund ratios for each item using transaction counts."""

    # Count transactions for each item and status
    # Use NumPy arrays to avoid issues with duplicate timestamp indices
    status_counts = pd.crosstab(
        df["item"].to_numpy(),
        df["status"].to_numpy(),
        rownames=["item"],
        colnames=["status"]
    )

    # Get SUCCESS and REFUND counts, using zero if a column is missing
    sales_count = status_counts.get(
        "SUCCESS", pd.Series(0, index=status_counts.index)
    )
    refund_count = status_counts.get(
        "REFUND", pd.Series(0, index=status_counts.index)
    )

    # Calculate refund ratios using the function from Task 3.2
    refund_ratios = compute_refund_ratio(
        sales_count.to_numpy(),
        refund_count.to_numpy()
    )

    # Create a DataFrame containing the results
    result = pd.DataFrame({
        "sales_count": sales_count,
        "refund_count": refund_count,
        "refund_ratio": refund_ratios
    })

    return result

## Test:

In [84]:
# Calculate refund ratios for each item
item_refund_ratios = calculate_item_refund_ratios(df)

# Display the results
print("Refund ratios per item:")
display(item_refund_ratios)

# Check the DataFrame dimensions
print(f"Result shape: {item_refund_ratios.shape}")

Refund ratios per item:


,sales_count,refund_count,refund_ratio
item,,,
Cappuccino,10,4,0.400000
Croissant,8,2,0.250000
Espresso,7,2,0.285714
Latte,15,3,0.200000
Muffin,5,3,0.600000
Sandwich,6,4,0.666667


Result shape: (6, 3)


## Discussion:

The refund ratio was calculated for each item using the number of
REFUND transactions divided by the number of SUCCESS transactions.

The existing NumPy function from Task 3.2 was reused, and epsilon
was included in the denominator to prevent division by zero.

CANCELLED transactions were excluded from both counts.

## Task 5: Full Workflow

### Task 5.1: Loading and Combining Transactions

This section combines the previously developed functions to load
transaction data from all log files.

Only transactions from active registers are included. The records
are then combined into a single Pandas DataFrame for further analysis.

In [85]:
# Find all log files recursively
all_log_files = find_log_files("data")

# Get the active register IDs from Task 2.2
active_ids = find_active_registers(store_config)

# Create a list to collect transactions from all files
all_transactions = []

# Read and parse each log file
for log_file in all_log_files:

    # Check whether the register is active
    if log_file.parent.name in active_ids:

        # Read the file with the appropriate encoding
        content, encoding = read_log_file(log_file)

        # Extract valid transactions
        records = parse_log(content)

        # Add the records to the combined list
        all_transactions.extend(records)

# Create one DataFrame containing all transactions
full_df = create_transaction_dataframe(all_transactions)

# Display the results
print(f"Total log files found: {len(all_log_files)}")
print(f"Total transactions: {len(all_transactions)}")
print(f"DataFrame shape: {full_df.shape}")

display(full_df.head())

Total log files found: 9
Total transactions: 720
DataFrame shape: (720, 5)


,register_id,item,price,currency,status
timestamp,,,,,
2026-10-15 07:00:37,POS_02,Croissant,3.46,EUR,SUCCESS
2026-10-15 07:00:50,POS_02,Sandwich,999.00,USD,SUCCESS
2026-10-15 07:01:13,POS_02,Espresso,3.03,USD,REFUND
2026-10-15 07:01:18,POS_01,Espresso,3.44,NOK,SUCCESS
2026-10-15 07:01:19,POS_03,Croissant,3.84,EUR,SUCCESS


### Task 5.2: Cleaning Price Outliers

In this step, we apply the NumPy outlier-cleaning function
developed in Task 3.1 to the complete transaction dataset.

Prices that deviate more than three standard deviations from
the mean are replaced with the median of valid prices.

The number of detected outliers is checked before and after cleaning.

In [86]:
# Create a copy to preserve the original DataFrame
cleaned_df = full_df.copy()

# Count the number of 999.00 price glitches before cleaning
outliers_before = (cleaned_df["price"] == 999.00).sum()

# Apply the NumPy outlier-cleaning function from Task 3.1
cleaned_df["price"] = remove_price_outliers(
    cleaned_df["price"].to_numpy()
)

# Count remaining 999.00 prices after cleaning
outliers_after = (cleaned_df["price"] == 999.00).sum()

# Display the results
print(f"999.00 prices before cleaning: {outliers_before}")
print(f"999.00 prices after cleaning: {outliers_after}")

# Check that the number of transactions remains unchanged
print(f"DataFrame shape after cleaning: {cleaned_df.shape}")

999.00 prices before cleaning: 17
999.00 prices after cleaning: 0
DataFrame shape after cleaning: (720, 5)


## Additional Testing Code:

In [87]:
# Compare original and cleaned prices
changed_prices = full_df["price"] != cleaned_df["price"]

# Count the number of modified transactions
print(f"Number of modified prices: {changed_prices.sum()}")

# Display some examples of modified prices
comparison = pd.DataFrame({
    "original_price": full_df.loc[changed_prices, "price"],
    "cleaned_price": cleaned_df.loc[changed_prices, "price"]
})

display(comparison.head(10))

Number of modified prices: 17


,original_price,cleaned_price
timestamp,,
2026-10-15 07:00:50,999.0,4.04
2026-10-15 07:01:39,999.0,4.04
2026-10-15 07:04:10,999.0,4.04
2026-10-15 07:10:54,999.0,4.04
2026-10-15 07:16:28,999.0,4.04
2026-10-15 07:29:25,999.0,4.04
2026-10-15 07:29:41,999.0,4.04
2026-10-15 07:31:56,999.0,4.04
2026-10-15 07:55:55,999.0,4.04


## Discussion:

The outlier-cleaning function was applied to all 720 transactions.
Before cleaning, 17 transactions contained the extreme price value
999.00. After applying the three-standard-deviation rule, no
999.00 prices remained.

The DataFrame retained all 720 transactions.

### Task 5.3: NOK Price Correction

In this step, incorrectly recorded NOK prices are corrected
by multiplying them by 10.0.

Only transactions with currency NOK are modified. Prices
recorded in EUR and USD remain unchanged.

In [88]:
# Identify transactions recorded in NOK
nok_mask = cleaned_df["currency"] == "NOK"

# Count NOK transactions before correction
nok_count = nok_mask.sum()

# Store original NOK prices for comparison
original_nok_prices = cleaned_df.loc[nok_mask, "price"].copy()

# Correct NOK prices by multiplying by 10
cleaned_df.loc[nok_mask, "price"] *= 10.0

# Display the results
print(f"Number of NOK transactions corrected: {nok_count}")
print(f"DataFrame shape: {cleaned_df.shape}")

Number of NOK transactions corrected: 204
DataFrame shape: (720, 5)


## Test:

In [89]:
# Compare NOK prices before and after correction
nok_comparison = pd.DataFrame({
    "original_price": original_nok_prices,
    "corrected_price": cleaned_df.loc[nok_mask, "price"]
})

# Display the first five corrected transactions
print("NOK price correction examples:")
display(nok_comparison.head())

# Verify that each NOK price was multiplied by 10
assert np.allclose(
    nok_comparison["corrected_price"],
    nok_comparison["original_price"] * 10.0
)

print("NOK price correction verified successfully!")

NOK price correction examples:


,original_price,corrected_price
timestamp,,
2026-10-15 07:01:18,3.44,34.4
2026-10-15 07:02:38,4.34,43.4
2026-10-15 07:02:46,2.98,29.8
2026-10-15 07:03:59,3.81,38.1
2026-10-15 07:04:16,3.70,37.0


NOK price correction verified successfully!


## Discussion:

NOK transactions were identified using Pandas Boolean indexing.
Their prices were multiplied by 10.0 to correct the recording error.

Prices in other currencies were left unchanged. The correction
was verified by comparing the original and updated NOK prices.

### Task 5.4: Generate Summary Reports

This section uses the cleaned transaction DataFrame to generate
three summary reports: hourly revenue, a transaction status
pivot table, and refund ratios per item.

The functions developed in Task 4 are reused to perform the analysis.

In [90]:
# Calculate hourly revenue for each item
final_hourly_revenue = calculate_hourly_revenue(cleaned_df)

# Create a pivot table of transaction prices by item and status
final_status_pivot = create_status_pivot_table(cleaned_df)

# Calculate refund ratios for each item
final_refund_ratios = calculate_item_refund_ratios(cleaned_df)

# Display the results
print("Hourly revenue:")
display(final_hourly_revenue.head(10))

print("Transaction status pivot table:")
display(final_status_pivot)

print("Refund ratios per item:")
display(final_refund_ratios)

Hourly revenue:


,item,timestamp,price
0,Cappuccino,2026-10-15 07:00:00,966.98
1,Cappuccino,2026-10-15 08:00:00,723.86
2,Cappuccino,2026-10-15 09:00:00,91.79
3,Croissant,2026-10-15 07:00:00,590.62
4,Croissant,2026-10-15 08:00:00,844.09
5,Croissant,2026-10-15 09:00:00,119.85
6,Espresso,2026-10-15 07:00:00,576.45
7,Espresso,2026-10-15 08:00:00,760.53
8,Espresso,2026-10-15 09:00:00,3.49
9,Latte,2026-10-15 07:00:00,1154.13


Transaction status pivot table:


status,CANCELLED,REFUND,SUCCESS
item,,,
Cappuccino,239.28,481.42,1061.93
Croissant,191.98,354.28,1008.30
Espresso,161.55,229.75,949.17
Latte,290.90,303.62,1270.21
Muffin,171.14,358.51,838.75
Sandwich,549.72,934.59,2006.92


Refund ratios per item:


,sales_count,refund_count,refund_ratio
item,,,
Cappuccino,73,25,0.342466
Croissant,70,18,0.257143
Espresso,91,23,0.252747
Latte,88,22,0.250000
Muffin,70,20,0.285714
Sandwich,88,26,0.295455


### Task 5.5: Save Output Files

The cleaned transactions and summary reports are exported
to CSV and JSON files.

All generated files are saved in the output directory
for submission and further analysis.

In [91]:
# Create an output directory if it does not already exist
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

# Save cleaned transactions to CSV
cleaned_df.to_csv(output_dir / "cleaned_transactions.csv")

# Save the transaction status pivot table to CSV
final_status_pivot.to_csv(output_dir / "status_pivot_table.csv")

# Save refund ratios per item to CSV
final_refund_ratios.to_csv(output_dir / "item_refund_ratios.csv")

# Convert hourly revenue into a list of dictionaries
hourly_records = final_hourly_revenue.to_dict(orient="records")

# Convert Pandas timestamps into strings for JSON serialization
for record in hourly_records:
    record["timestamp"] = record["timestamp"].isoformat()

# Save hourly revenue to JSON
with (output_dir / "hourly_revenue.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(hourly_records, file, indent=4)

print("All output files saved successfully!")

All output files saved successfully!


## Test:

In [92]:
# List the expected output files
expected_files = [
    "cleaned_transactions.csv",
    "hourly_revenue.json",
    "status_pivot_table.csv",
    "item_refund_ratios.csv"
]

# Verify that each output file exists
for filename in expected_files:
    file_path = output_dir / filename

    print(f"{filename}: {'Found' if file_path.is_file() else 'Missing'}")

cleaned_transactions.csv: Found
hourly_revenue.json: Found
status_pivot_table.csv: Found
item_refund_ratios.csv: Found


In [93]:
# Read the exported CSV files
saved_transactions = pd.read_csv(
    output_dir / "cleaned_transactions.csv"
)

saved_pivot = pd.read_csv(
    output_dir / "status_pivot_table.csv"
)

saved_refunds = pd.read_csv(
    output_dir / "item_refund_ratios.csv"
)

# Read the exported JSON report
with (output_dir / "hourly_revenue.json").open(
    "r", encoding="utf-8"
) as file:
    saved_hourly = json.load(file)

# Verify the exported data
assert len(saved_transactions) == 720
assert saved_pivot.shape == (6, 4)
assert saved_refunds.shape == (6, 4)
assert len(saved_hourly) == len(final_hourly_revenue)
assert (saved_transactions["price"] != 999.0).all()

print("All exported files passed validation!")

All exported files passed validation!


## Discussion:

The cleaned dataset was used to generate hourly transaction
summaries, a status pivot table, and item-level refund ratios.

All four required output files were successfully exported.
The dataset contains 720 transactions, with outlier prices
cleaned and NOK prices corrected.

The reported price totals combine different currencies without
exchange-rate conversion, as no exchange rates were provided.
Therefore, these totals should not be interpreted as financial
revenue in a single currency.